# 01. 제조 산업 AI 활용 Overview

> **Day 01 — 제조 시계열 AI**
> - 제조 산업에서 AI가 어디에 쓰이는지 이해하고, 현실 문제상황에서 발생가능한 상황에 대해 코드로 직접 체감한다.
>

---

> **실습 안내**
> `"""채워넣기"""` 가 적힌 셀은 코드 대부분이 이미 있고, `"""채워넣기"""`로 표시된 부분만 채우면 됩니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차
1. Colab 실습 환경 준비
2. 제조 센서 데이터 검토 — 사출성형기
3. 난제를 코드로 체감하기 — 불균형과 도메인 시프트

> ℹ️ Day 1 실습에는 **API 키가 필요 없습니다.** (Day 2 LLM 실습부터 필요합니다)

**오늘의 진행 방식**

- 이론과 Colab 실습을 오가며 진행합니다. 노트북 5권을 순서대로 다룹니다.
- `"""채워넣기"""` 가 적힌 셀은 코드 대부분이 있고, 표시된 부분만 채웁니다.
- 각 노트북 마지막의 **Self-check** 로 스스로 점검합니다.

---
## 1. Colab 실습 환경 준비

> 오늘 딥러닝 실습은 무료 **T4 GPU** 기준으로 설계되어 있습니다.

**GPU 런타임으로 변경하는 방법**

1. 상단 메뉴에서 **런타임 → 런타임 유형 변경** 을 엽니다.
2. "하드웨어 가속기"에서 **T4 GPU** 를 선택하고 저장을 누릅니다.
3. 우측 상단에 RAM/디스크 게이지가 다시 뜨면 연결 완료입니다.

변경하면 세션이 초기화되므로 **노트북 시작 시점에 먼저** 바꿔 두는 것이 좋습니다.
아래 셀로 GPU가 잡혔는지 확인합니다.

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

In [ ]:
# GPU 동작 확인 — 행렬곱이 어느 장치에서 도는지 본다
"""채워넣기"""
import torch

x = torch.randn(2000, 2000)
y = torch.randn(2000, 2000)

import time
t0 = time.time()
z = (x.to("""채워넣기""") @ y.to("""채워넣기"""))
print(f"장치: {DEVICE} | 2000x2000 행렬곱: {time.time() - t0:.3f}초")
print(f"결과 shape: {z.shape}, 합계: {z.sum().item():.1f}")

In [ ]:
# 재현성 확인 — 시드를 고정하면 난수도 매번 같다
np.random.seed(SEED)
a = np.random.rand(3)
np.random.seed(SEED)
b = np.random.rand(3)
print("첫 번째:", a.round(4))
print("두 번째:", b.round(4))
print("동일 여부:", np.allclose(a, b), "— 오늘 모든 실습은 SEED=42로 재현됩니다")

---
## 2. 제조 센서 데이터 검토 — 사출성형기

> 시계열 데이터는 **설비의 심전도**입니다. 파형을 보면 설비 상태를 알 수 있습니다.

첫 데이터로 **사출성형기**를 봅니다. 플라스틱을 녹여 금형(틀)에 쏘아 넣고
식혀서 꺼내는 설비로, 한 번의 생산 단위를 **사이클**이라 부릅니다.

| 센서 | 의미 | 신호 패턴 |
|---|---|---|
| `mold_temp` | 금형 온도(℃) | 가동 초반 warm-up 램프(약 45℃→65℃) + 사이클마다 반복되는 소폭 진동 |
| `inj_pressure` | 사출압력(bar) | 사이클마다 사출 순간의 뾰족한 피크 + 뒤이은 보압(holding) 구간 |
| `screw_rpm` | 스크류 회전수 | 사이클 초반 사출 구간에서만 상승, 나머지 구간은 낮게 유지 |

**데이터 설명(Data Description)**

| 항목 | 내용 |
|---|---|
| 샘플링 | 1초 간격 |
| 규모 | 120사이클 × 사이클당 약 30초 ≈ **3,600행**(약 1시간 분량) |
| 구조 | `df`(센서 3개, X 역할) + `labels`(시점별 0/1, y 역할) |

`labels`는 사이클 단위가 아니라 **시점 단위** 라벨입니다. 3,600개 시점 하나하나에
정상/이상 정답이 붙어 있다는 뜻입니다. 이상 유형은 세 가지가 섞여 있고, 전체 비율은 약 1~2%입니다.

| 유형 | 개수·길이 | 특징 |
|---|---|---|
| **Point** | 4곳, 1시점 | `inj_pressure`에 순간 스파이크 — 눈에 잘 띕니다 |
| **Contextual** | 1곳, 12초 | 냉각 구간인데 `mold_temp`가 상승 — **값 자체는 정상 범위**라 눈으로는 놓치기 쉽습니다 |
| **Collective** | 1곳, 25초 | `inj_pressure`·`screw_rpm`이 동시에 리듬을 잃는 구간 |

여기에 더해 라벨과 무관하게 `inj_pressure`엔 **결측 블록**, `mold_temp`엔 **홀드값**(센서 고착)도
함께 심어 두었습니다. 현장에서 센서가 실제로 겪는 문제상황입니다.

In [ ]:
# 사출성형기 센서 데이터 생성 — 1초 샘플링, 약 1시간 분량
"""채워넣기"""
df, labels = mfg_datagen.gen_injection_molding(n_cycles="""채워넣기""", seed=SEED, return_labels=True)
print(f"shape: {df.shape} | 기간: {df.index[0]} ~ {df.index[-1]}")
df.head()

In [ ]:
# 기술통계 — 각 센서의 스케일이 서로 크게 다르다는 점을 살펴본다
print(df.describe().round(2).to_string())

In [ ]:
# 세 센서를 위아래로 나란히 그려 전체 흐름을 본다
"""채워넣기"""
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, col in zip(axes, df.columns):
    ax.plot(df.index, df["""채워넣기"""], lw=0.6)
    ax.set_ylabel(col)
axes[0].set_title("Injection Molding — 3 sensors")
plt.tight_layout()
plt.show()

In [ ]:
# 확대해서 사이클 구조를 본다 — 앞쪽 3분(사이클 약 6개) * 데이터는 1초 단위로 수집
zoom = df.iloc[:180]
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(zoom.index, zoom["inj_pressure"], label="inj_pressure")
ax.plot(zoom.index, zoom["mold_temp"], label="mold_temp")
ax.set_title("Zoom: first 3 minutes (cycle structure)")
ax.legend()
plt.tight_layout()
plt.show()
print("사출 피크가 일정한 간격으로 반복됩니다 — 이 규칙성이 곧 '정상'의 정의입니다.")

### 관찰 포인트

- **warm-up 램프**: 가동 초반 금형 온도가 완만하게 오릅니다. 이 구간은 "정상이지만 특수한" 구간입니다.
- **사이클 주기성**: 압력 피크가 일정한 간격으로 계속 반복됩니다.
- 그런데 그래프 중간에 **선이 끊긴 구간**(결측)과 **일자로 눌린 구간**(홀드값)이 보입니다.

> **홀드값(센서 고착)** 은 *가동이 멈춘 상태* 입니다.
> 통신은 살아 있는데 값만 갱신되지 않는, 제조 데이터 특유의 특징입니다.

In [ ]:
# 결측 블록과 홀드값(동일값 반복)이 실제로 있는지, 몇 포인트인지 확인
na_cnt = df.isna().sum()
n_missing = int(na_cnt.sum())
print(f"결측 포인트: 총 {n_missing}개")
print(na_cnt[na_cnt > 0].to_string())

same_as_prev = (df["mold_temp"].diff() == 0)
n_hold = int(same_as_prev.sum())
max_run = (same_as_prev.groupby((~same_as_prev).cumsum()).cumsum()).max()
print(f"\n홀드값(mold_temp 동일값 반복) 포인트: 총 {n_hold}개, 최장 연속 {int(max_run)}초")
print("— 1초 샘플링 센서에서 이 길이는 물리적으로 어렵습니다")

---
## 3. 제조 AI 에서 발생가능한 난제를 코드로 검토해보기

### 3-1. 극심한 클래스 불균형

> *1만 개 중 3개의 불량 — 전부 정상이라 답해도 정확도 99.97%.*

방금 만든 데이터에는 이상 구간 라벨이 함께 들어 있습니다.
실제 공장의 이상 비율이 어느 수준인지, 그때 정확도(Accuracy)가 왜 무의미해지는지 숫자로 확인합니다.

In [ ]:
# 이상 라벨 비율 확인 — 정상과 이상이 몇 대 몇인가
"""채워넣기"""
n_total = len(labels)
n_anom = """채워넣기"""
print(f"전체 시점 : {n_total:,}")
print(f"이상 시점 : {n_anom:,}")
print(f"이상 비율 : {100 * n_anom / n_total:.2f}%")

In [ ]:
# "전부 정상"이라고만 답하는 모델의 정확도를 계산해 본다
"""채워넣기"""
pred_all_normal = """채워넣기"""          # 모든 시점을 정상(0)으로 예측
accuracy = ("""채워넣기""").mean()
print(f"아무것도 탐지하지 않는 모델의 정확도: {100 * accuracy:.2f}%")
print("→ 정확도만 보면 훌륭한 모델처럼 보입니다. 하지만 이상은 단 한 건도 잡지 못했습니다.")

In [ ]:
# 혼동행렬(Confusion Matrix)로 방금 본 정확도의 함정을 다시 확인한다
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels.values, pred_all_normal, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: Normal", "Pred: Anomaly"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True: Normal", "True: Anomaly"])
ax.set_title("Confusion Matrix — 'always normal' model")
plt.tight_layout()
plt.show()

print(f"TN(정상→정상) {tn:,} | FP(정상→이상 오탐) {fp:,}")
print(f"FN(이상→정상, 놓친 이상) {fn:,} | TP(이상→이상 적중) {tp:,}")
print("→ 'Pred: Anomaly' 열이 통째로 비어 있습니다. 정확도는 높아도 이상 탐지 능력은 0입니다.")

In [ ]:
# 이상 시점이 실제로 어디에 있는지 압력 신호 위에 표시
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(df.index, df["inj_pressure"], lw=0.6, label="inj_pressure")
anom_idx = df.index[labels.values == 1]
ax.scatter(anom_idx, df.loc[anom_idx, "inj_pressure"], color="red", s=12,
           zorder=3, label="anomaly")
ax.set_title("Anomaly locations (red)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"빨간 점 {n_anom}개 — 이 소수를 잡는 것이 이상탐지의 전부입니다.")

**왜 문제인가**

- 정확도(Accuracy)는 데이터가 많은 쪽, 즉 정상 클래스로 쏠리는 지표입니다. 불균형 데이터에서는 적절한 평가지표로 활용할 수 없습니다.
- 향후, NB04에서는 이 지표를 Precision·Recall·PR-AUC로 평가지표를 바꿉니다.
- 임계값(Threshold) 은 알람이 얼마나 예민하게 반응할지 정하는 값입니다 — 예민하게 잡을수록 오탐이, 둔감하게 잡을수록 미검출이 늘어납니다.

> **현장 노트**
> - 오탐(False Alarm) 한 번은 라인을 세우는 비용이고, 미검출(Miss) 한 번은 고객 클레임과 리콜로 번지는 비용입니다.
> - 크기도 다르고 공정마다 비율도 다르니, 두 비용을 같은 저울에 놓을 수 없습니다.
> - 결국 임계값은 데이터 과학이 아니라 현장의 비용 구조가 정합니다.

### 6-2. 도메인 시프트 미리보기

> 같은 기종의 설비 두 대. 데이터도 같을까요?

NB03에서 본격적으로 다룰 **항공기 엔진 데이터(C-MAPSS)** 를 잠깐 미리 봅니다.
FD001과 FD003은 **같은 엔진**을 **다른 운전 조건**에서 기록한 데이터셋입니다.

In [ ]:
# 같은 엔진, 다른 운전 조건의 두 데이터셋 로드 (FD001 vs FD003)
"""채워넣기"""
train_a, _, _ = loaders.load_cmapss("""채워넣기""")
train_b, _, _ = loaders.load_cmapss("""채워넣기""")
print(f"설비 A(FD001): {train_a.shape} | 설비 B(FD003): {train_b.shape}")
print("컬럼 구성은 완전히 동일합니다:", list(train_a.columns[:8]), "...")

In [ ]:
# 같은 센서(s2, s11)의 분포를 겹쳐 그려 본다
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, s in zip(axes, ["s2", "s11"]):
    ax.hist(train_a[s], bins=60, alpha=0.6, label="FD001 (A)", density=True)
    ax.hist(train_b[s], bins=60, alpha=0.6, label="FD003 (B)", density=True)
    ax.set_title(f"sensor {s} distribution")
    ax.legend()
plt.tight_layout()
plt.show()
print("같은 기종·같은 센서인데 분포의 중심이 어긋나 있습니다.")

In [ ]:
# 눈이 아니라 숫자로 — 두 설비의 센서 평균이 얼마나 어긋났는지 표준편차 단위로 계산
shift = pd.DataFrame({
    "mean_A": train_a[["s2", "s3", "s11", "s15"]].mean(),
    "mean_B": train_b[["s2", "s3", "s11", "s15"]].mean(),
    "std_A": train_a[["s2", "s3", "s11", "s15"]].std(),
})
shift["shift(σ)"] = ((shift["mean_B"] - shift["mean_A"]) / shift["std_A"]).round(2)
print(shift.round(2).to_string())

flagged = shift[shift["shift(σ)"].abs() > 1]
if len(flagged):
    names = ", ".join(flagged.index)
    print(f"\nshift(σ)가 1을 넘는 센서: {names}")
    print(f"→ A 기준 정상 범위를 벗어났습니다 — B의 평균적인 {names} 값이 A 입장에서는 이미 \"흔치 않은 값\"으로 보입니다.")
else:
    print("\n이번 실행에서는 4개 센서 모두 shift(σ)가 1 미만입니다 — 평균만 보면 A·B가 비슷해 보입니다.")
print("도메인 시프트는 모든 센서에 균등하게 오지 않습니다 — 운전 조건에 민감한 센서만 골라서 어긋납니다.")

**이것이 난제 ③ Concept Drift & Sensor Shift 입니다**

- 도메인 시프트는 센서마다 다르게 옵니다. 운전 조건에 민감한 센서만 크게 어긋나고, 나머지는 별 차이가 없습니다.
- A 설비로 학습한 모델을 B 설비에 그대로 쓰면, 모델이 정상이라고 배운 기준 자체가 맞지 않습니다.
- 부품 교체나 공정 조건 변경, 계절 변화도 같은 문제를 일으킵니다.

> **현장 노트**
> - F1 0.95를 찍은 논문 모델이 현장에서 무너지는 흔한 이유는 설비마다 센서 위치와 샘플링 주기가 다르기 때문입니다.
> - 같은 기종이어도 A동 3호기와 B동 7호기는 사실상 다른 데이터입니다.

---
## Self-check

### Q1. AI Adopter로서 제조 기업의 AI 프로젝트는 무엇으로 평가받아야 합니까?

<details>
<summary>정답 보기</summary>

- 모델 성능 지표가 아니라 **물리적 P&L 개선**으로 평가받습니다.
- "수율 0.1% 향상", "에너지 비용 5% 절감"처럼 손익의 언어로 번역되어야 합니다.
- 같은 이유로, 기술 선택도 "가장 최신"이 아니라 "비용 대비 효과"가 기준이 됩니다.

</details>

---

### Q2. 이상 비율 1%인 데이터에서 정확도(Accuracy) 99%는 무엇을 의미합니까?

<details>
<summary>정답 보기</summary>

- 거의 아무 정보도 주지 못합니다. "전부 정상" 예측만으로도 99%가 나오기 때문입니다.
- 불균형 데이터에서는 Precision·Recall·PR-AUC 같은 지표로 성적표를 바꿔야 합니다.
- 최종 판단에는 오탐 비용(라인 정지)과 미검출 비용(클레임·리콜)의 비대칭까지 반영해야 합니다.

</details>

---

### Q3. 신규 라인 증설 직후에 이상탐지 모델을 만들기 어려운 근본 이유는 무엇입니까?

<details>
<summary>정답 보기</summary>

- 학습에 쓸 데이터가 없는 **Cold-Start** 상태이기 때문입니다.
- 이상 데이터는 물론이고 "정상이 무엇인지"를 정의할 정상 데이터조차 부족합니다.
- 라벨 없이 표현을 먼저 배우는 자기지도학습(SSL)이 이 문제의 돌파구이며, NB05에서 다룹니다.

</details>

---

### Q4. [현장 판단] 같은 기종 설비 10대에 단일 모델을 배포했더니 3대에서만 오탐이 쏟아집니다. 어디부터 의심해야 합니까?

<details>
<summary>정답 보기</summary>

- 모델보다 **데이터 분포의 차이**부터 의심합니다 — 센서 설치 위치, 샘플링 주기, 부품 교체 이력.
- 오늘 본 FD001/FD003 히스토그램처럼, 같은 기종도 운전 조건이 다르면 분포가 어긋납니다.
- 3대의 데이터 분포를 나머지와 비교해 시프트를 확인한 뒤, 도메인 적응 또는 설비별 보정을 검토합니다.

</details>


---
## 다음 노트북 예고 — NB02. 시계열 데이터의 이해와 실습 토대

방금 그래프에서 본 **결측 블록**, 그냥 지나칠 수 없습니다.
NB02에서는 시계열 데이터를 "순서에 정보가 있는 데이터"로 정의하고,
리샘플링 → 결측 처리 → 슬라이딩 윈도우 → 시간 기반 분할까지,
이후 모든 실습이 딛고 설 토대를 만듭니다.

> 특히 **시간 기반 분할**에서는 랜덤 셔플이 만드는 **데이터 누수**를
> 틀린 코드와 옳은 코드로 나란히 실행해 눈으로 확인합니다.